In [ ]:
import nglview as nv

# Create widget without default representation
view = nv.NGLWidget()

# Add component with no default representation
view.add_component("/home/ilnitsky/NPM/fold_res/Myxine_glutinosa_1/Myxine_glutinosa_1/unrelaxed_model_1_pred_0.pdb", default_representation=False)

# Add cartoon with bfactor coloring (rainbow gradient)
comp.add_cartoon(color_scheme='bfactor')

# Set background color to white
view.background = 'white'

# Display the view
view


In [ ]:
import nglview as nv

view = nv.NGLWidget()
view.add_component("/home/ilnitsky/NPM/fold_res/Myxine_glutinosa_1/Myxine_glutinosa_1/unrelaxed_model_1_pred_0.pdb")

# Remove default representation
view.clear_representations()

# Use a basic cartoon representation with secondary structure coloring scheme
view.add_cartoon(selection='protein')

# Override the default sstruc coloring with custom coloring
# Blue for structured regions (helices and sheets)
view.add_cartoon(selection='helix or sheet', color='blue')

# Yellow to red gradient for unstructured regions (coil/loops)
view.add_cartoon(selection='coil or loop', color='resname', scheme='yelloworange')

# Set background color to white
view.background = 'white'

# Display the view
view.display()


In [ ]:
import nglview as nv

view = nv.NGLWidget()
view.add_component("/home/ilnitsky/NPM/fold_res/Myxine_glutinosa_1/Myxine_glutinosa_1/unrelaxed_model_1_pred_0.pdb")

# Remove default representation
view.clear_representations()

# Add beta sheets in dark blue
view.add_representation('cartoon', selection='sheet', color='blue')

# Add alpha helices in light blue/cyan
view.add_representation('cartoon', selection='helix', color='lightblue')

# Add unstructured regions (coil/loop) with yellow to red gradient
view.add_representation('cartoon', selection='coil or loop', color='resname', scheme='yelloworange')

# Set background color to white
view.background = 'white'

# Adjust parameters for thin cartoon representation
view._remote_call('setParameters', target='Widget', args=[{'quality': 'medium'}])

# Display the view
view.display()


In [ ]:
import nglview as nv

view = nv.NGLWidget()
view.add_component("/home/ilnitsky/NPM/fold_res/Myxine_glutinosa_1/Myxine_glutinosa_1/unrelaxed_model_1_pred_0.pdb")

# Remove default representation
view.clear_representations()

# Add structured regions (helices and sheets) in blue
view.add_representation('cartoon', selection='helix or sheet', color='blue')

# Add unstructured regions (coil/loop) with yellow to red gradient
# Using a colorscheme that transitions from yellow to red
view.add_representation('cartoon', selection='coil or loop', color='resname', scheme='yellowerange')

# Optional: show protein surface as transparent
view.add_representation('surface', opacity=0.5, colorScheme='residueindex')

view.display()


In [8]:
from Bio.PDB import PDBParser, Superimposer, PDBIO
import glob
import os

def get_ca_atoms_by_residue(structure):
    ca_atoms = {}
    for model in structure:
        for chain in model:
            for residue in chain:
                if "CA" in residue:
                    ca_atoms[residue.get_id()] = residue["CA"]
    return ca_atoms

# Parse structures
parser = PDBParser()
pdb_dir = "/home/ilnitsky/NPM/pdbs/Artrhropoda"
pdb_files = sorted(glob.glob(os.path.join(pdb_dir, "*.pdb")))

parser = PDBParser()
structures = [parser.get_structure(f"struct{i+1}", pdb_file) for i, pdb_file in enumerate(pdb_files)]

# Get CA atoms by residue for each structure
ca_dicts = [get_ca_atoms_by_residue(struct) for struct in structures]

# Find common residue IDs
common_residues = set(ca_dicts[0].keys())
for ca_dict in ca_dicts[1:]:
    common_residues &= set(ca_dict.keys())

# Sort residue IDs for consistent order
common_residues = sorted(common_residues)

# Build atom lists for alignment
atoms_list = []
for ca_dict in ca_dicts:
    atoms = [ca_dict[res_id] for res_id in common_residues]
    atoms_list.append(atoms)

# Align all structures to the first one
ref_atoms = atoms_list[0]
superimposer = Superimposer()
for mobile_atoms in atoms_list[1:]:
    superimposer.set_atoms(ref_atoms, mobile_atoms)
    superimposer.apply(mobile_atoms)

output_paths = []
for i, struct in enumerate(structures):
    io = PDBIO()
    io.set_structure(struct)
    out_path = f"aligned_struct_{i+1}.pdb"
    io.save(out_path)
    output_paths.append(out_path)

import nglview as nv

view = nv.NGLWidget()
for path in output_paths:
    view.add_component(path)
view


NGLWidget()

In [10]:
print("\nPairwise RMSD to reference structure:")
for i, mobile_atoms in enumerate(atoms_list[1:], start=2):
    superimposer.set_atoms(ref_atoms, mobile_atoms)
    rms = superimposer.rms
    print(f"Reference vs struct{i}: RMSD = {rms:.3f} Å")


Pairwise RMSD to reference structure:
Reference vs struct2: RMSD = 22.878 Å
Reference vs struct3: RMSD = 13.239 Å
Reference vs struct4: RMSD = 32.655 Å
Reference vs struct5: RMSD = 21.854 Å
Reference vs struct6: RMSD = 24.480 Å


In [5]:
from Bio.PDB import PDBParser, Superimposer, PDBIO, PPBuilder
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment
from Bio import AlignIO
from Bio.PDB.Polypeptide import three_to_one
import glob
import os

import re
import requests


def get_uniprot_id(filename):
    match = re.search(r"AF-([A-Z0-9]+)-", os.path.basename(filename))
    return match.group(1) if match else None

def fetch_uniprot_sequence(uniprot_id):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        print(f"Failed to fetch sequence for {uniprot_id}")
        return None

def get_ca_atoms_by_residue(structure):
    ca_atoms = {}
    for model in structure:
        for chain in model:
            for residue in chain:
                if "CA" in residue:
                    ca_atoms[residue.get_id()] = residue["CA"]
    return ca_atoms

def get_full_sequence(structure):
    """Extract the full sequence of the structure."""
    seq = []
    for model in structure:
        for chain in model:
            for residue in chain:
                try:
                    seq.append(three_to_one(residue.get_resname()))
                except KeyError:
                    seq.append('X')  # Non-standard residues
    return ''.join(seq), [residue.get_id() for residue in chain]

def get_msa_from_structural_alignment(structures, common_residues, uniprot_ids):
    """Create MSA based on common residues from structural alignment."""
    msa = MultipleSeqAlignment([])
    for i, struct in enumerate(structures):
        full_seq, res_ids = get_full_sequence(struct)
        # Create aligned sequence with gaps
        aligned_seq = []
        for res_id in common_residues:
            if res_id in res_ids:
                idx = res_ids.index(res_id)
                aligned_seq.append(full_seq[idx])
            else:
                aligned_seq.append('-')  # Gap for non-aligned residue
        # Use UniProt ID instead of generic pdb label
        uniprot_id = uniprot_ids[i] if i < len(uniprot_ids) else f"pdb{i+1}"
        seq_record = SeqRecord(Seq(''.join(aligned_seq)), id=uniprot_id, description=f"Structure {i+1}")
        msa.append(seq_record)
    return msa

# Parse structures
pdb_dir = "/home/ilnitsky/NPM/pdbs/Artrhropoda"
output_dir = os.path.join(os.getcwd(), "aligned_structures")
os.makedirs(output_dir, exist_ok=True)
pdb_files = sorted(glob.glob(os.path.join(pdb_dir, "*.pdb")))
if not pdb_files:
    raise ValueError(f"No PDB files found in {pdb_dir}")

parser = PDBParser(QUIET=True)
structures = [parser.get_structure(f"struct{i+1}", pdb_file) for i, pdb_file in enumerate(pdb_files)]

# Get CA atoms by residue for each structure
ca_dicts = [get_ca_atoms_by_residue(struct) for struct in structures]

# Find common residue IDs
common_residues = set(ca_dicts[0].keys())
for ca_dict in ca_dicts[1:]:
    common_residues &= set(ca_dict.keys())

if not common_residues:
    raise ValueError("No common residues found across all structures")

# Sort residue IDs for consistent order
common_residues = sorted(common_residues)

# Build atom lists for alignment
atoms_list = []
for ca_dict in ca_dicts:
    atoms = [ca_dict[res_id] for res_id in common_residues]
    atoms_list.append(atoms)

# Align all structures to the first one
ref_atoms = atoms_list[0]
superimposer = Superimposer()

output_paths = []
for i, (mobile_atoms, struct) in enumerate(zip(atoms_list[1:], structures[1:])):
    try:
        superimposer.set_atoms(ref_atoms, mobile_atoms)
        superimposer.apply(struct.get_atoms())
        io = PDBIO()
        io.set_structure(struct)
        uniprot_id = get_uniprot_id(pdb_files[i+1])
        out_path = os.path.join(output_dir, f"aligned_{uniprot_id}.pdb")
        io.save(out_path)
        output_paths.append(out_path)
        # Fetch and save sequence
        seq = fetch_uniprot_sequence(uniprot_id)
        if seq:
            with open(os.path.join(output_dir, f"{uniprot_id}.fasta"), "w") as f:
                f.write(seq)
    except Exception as e:
        print(f"Error aligning structure {i+2}: {e}")

# Save reference structure
io = PDBIO()
io.set_structure(structures[0])
uniprot_id = get_uniprot_id(pdb_files[0])
out_path = os.path.join(output_dir, f"aligned_{uniprot_id}.pdb")
io.save(out_path)
output_paths.insert(0, out_path)
seq = fetch_uniprot_sequence(uniprot_id)
if seq:
    with open(os.path.join(output_dir, f"{uniprot_id}.fasta"), "w") as f:
        f.write(seq)

# Collect UniProt IDs for all structures
uniprot_ids = [get_uniprot_id(pdb_file) for pdb_file in pdb_files]

# Generate MSA with UniProt IDs
msa = get_msa_from_structural_alignment(structures, common_residues, uniprot_ids)
msa_path = os.path.join(output_dir, "aligned_sequences_structural.fasta")
AlignIO.write(msa, msa_path, "fasta")
print(f"Multiple sequence alignment saved to {msa_path}")

# Optional: Visualize with nglview
try:
    import nglview as nv
    view = nv.NGLWidget()
    for path in output_paths:
        view.add_component(path)
    view.display()
except ImportError:
    print("nglview not installed; skipping visualization")

Multiple sequence alignment saved to /home/ilnitsky/NPM/notebooks/aligned_structures/aligned_sequences_structural.fasta


In [23]:



import nglview as nv

view = nv.NGLWidget()
view.add_component("/home/ilnitsky/NPM/fold_res/Myxine_glutinosa_1/Myxine_glutinosa_1/unrelaxed_model_1_pred_0.pdb")

# Remove default representation
view.clear_representations()

view.add_cartoon('protein',color_scheme='bfactor')


# Set background color to white
view.background = 'light gray'


# Display the view
view.display()

NGLWidget(background='light gray')

In [34]:

 

import nglview as nv

view = nv.NGLWidget()
view.add_component("/home/ilnitsky/NPM/fold_res/Ptychodera_flava/Ptychodera_flava/ranked_4.pdb", default_representation=False)

# First add the component, then remove default representation
comp = view.component_0
comp.clear()

# Add cartoon with bfactor coloring (rainbow gradient)
comp.add_cartoon(color_scheme='bfactor')

# Set background color to white
# view.background = 'white'

# Display the view
view

NGLWidget()